# 02 · Pré-processamento

Define e valida separações, transformações e remoções das features

In [1]:
# TODO: transformar decisões aprovadas em funções reutilizáveis de src/.

In [ ]:
from pathlib import Path
import joblib
import json

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score, adjusted_rand_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

In [ ]:
# Definição dos Diretórios
RAW = Path("../data/raw/Steel_industry_data.csv")
PROCESSED = Path("../data/processed")
MODELS = Path("../models")
METADATA = Path("../metadata")

### 1) Importação dos Dados

In [ ]:
df = pd.read_csv(RAW)
target_column = "Load_Type"

df["date"] = pd.to_datetime(df["date"], dayfirst=True)
df = df.sort_values(by="date").reset_index(drop=True)

df.head()

,date,Usage_kWh,Lagging_Current_Reactive.Power_kVarh,Leading_Current_Reactive_Power_kVarh,CO2(tCO2),Lagging_Current_Power_Factor,Leading_Current_Power_Factor,NSM,WeekStatus,Day_of_week,Load_Type
0,2018-01-01 00:00:00,3.42,3.46,0.0,0.0,70.30,100.0,0,Weekday,Monday,Light_Load
1,2018-01-01 00:15:00,3.17,2.95,0.0,0.0,73.21,100.0,900,Weekday,Monday,Light_Load
2,2018-01-01 00:30:00,4.00,4.46,0.0,0.0,66.77,100.0,1800,Weekday,Monday,Light_Load
3,2018-01-01 00:45:00,3.24,3.28,0.0,0.0,70.28,100.0,2700,Weekday,Monday,Light_Load
4,2018-01-01 01:00:00,3.31,3.56,0.0,0.0,68.09,100.0,3600,Weekday,Monday,Light_Load


### 2) Separação dos Dados

Os dados serão dividos em treino e teste por meio de um split temporal de 80 % de treino e 20% de teste 

In [ ]:
# Definição do Eixo X e do Eixo Y
X = df.drop(columns=[target_column])
y = df[target_column].copy()

# Split temporal
cut = int(len(df) * 0.8)

X_train, X_test = X.iloc[:cut].copy(), X.iloc[cut:].copy()
y_train, y_test = y.iloc[:cut].copy(), y.iloc[cut:].copy()

print(f'Treino: {X_train['date'].min()} -> {X_train['date'].max()}')
print(f'Teste: {X_test['date'].min()} -> {X_test['date'].max()}')

Treino: 2018-01-01 00:00:00 -> 2018-10-19 23:45:00
Teste: 2018-10-20 00:00:00 -> 2018-12-31 23:45:00


### 3) Seleção das Features

Será adaptado e selecionado as features ideais para avaliar os modelos

In [ ]:
def time_feature_adapt(data: pd.DataFrame) -> pd.DataFrame:
    out = data.copy()

    out["hour"] = out["date"].dt.hour + out["date"].dt.minute / 60
    out["month"] = out["date"].dt.month
    out["dayofweek_num"] = out["date"].dt.day_of_week
    out['NSM_sin'] = np.sin(2 * np.pi * out['NSM'] / 86400)
    out['NSM_cos'] = np.cos(2 * np.pi * out['NSM'] / 86400)
    
    out = out.drop(columns=["date"])
    return out

X_train_featured = time_feature_adapt(X_train)
X_test_featured = time_feature_adapt(X_test)

In [ ]:
FEATURE_NUMERIC = [
    'Usage_kWh', # feature central
    'Lagging_Current_Reactive.Power_kVarh',  # r=0.9 com o 'Usage_kWh'
    'Leading_Current_Reactive_Power_kVarh',  # r=-0.32 com o 'Usage_kWh'
    'Lagging_Current_Power_Factor', # r=0.39 com o 'Usage_kWh'

    "NSM_sin", "NSM_cos", # substituem o NSM cru, capturam a ciclicidade que o Pearson não prevê
    'hour', 'month', 'dayofweek_num' # adaptação do calendário
]

FEATURE_CLUSTERS_CATEGORICAL = ['Day_of_week']

FEATURE_DISCARTED = [
    "CO2(tCO2)", # (alta correlação com 'Usage_kwh' -> redundância quade perfeita); mantida em artifacts para enriquerecer o relatório dos clusters
    "Leading_Current_Power_Factor", # variância quase nula, inútil para distância, mas útil para o relatório
    "WeekStatus", # redundante com o Day_of_week
]

FEATURE_DELETED = [
    "date",  # explodida em hour/month/dayofweek_num
    "NSM",   # substituída por NSM_sin / NSM_cos
]

### 4) Pré-Processamento

In [ ]:
preprocessor = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), FEATURE_NUMERIC),
        ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False), FEATURE_CLUSTERS_CATEGORICAL)
    ]
)

pipeline = Pipeline(steps=[
    ("prep", preprocessor),
    ("pca",  PCA(n_components=0.90, random_state=42)),
])

X_train_processed = pipeline.fit_transform(X_train_featured)
X_test_processed = pipeline.transform(X_test_featured) # é utilizado somente 'transform' pata evitar vazamento de dados do teste para o treino

components = pipeline.named_steps["pca"].n_components_

print(f"O Pipeline manteve {components} componetes para explicar 90% da variância")

O Pipeline manteve 7 componetes para explicar 90% da variância


#### 4.1) Correlações

Dos Clusters apresentados, será availado o quanto cada um se correlaciona entre si,
teria que ser uma correlação 0 para destacar que cada um agrupa classificações diferentes

In [ ]:
comp_df = pd.DataFrame(X_train_processed, columns=[f"PC{i+1}" for i in range(components)])

corr_pos = comp_df.corr().abs()

upper = corr_pos.where(np.triu(np.ones(corr_pos.shape), k=1).astype(bool))
print(f"Correlação máxima entre componentes (deve ser ~0): {upper.stack().max().round(6)}")

Correlação máxima entre componentes (deve ser ~0): 0.0


#### 4.2) Avaliação por Perfil

In [ ]:
km_final = KMeans(n_clusters=3, random_state=42, n_init=10).fit(X_train_processed)

df_profile = X_train.copy()
df_profile["cluster"] = km_final.labels_


main_profile = df_profile.groupby("cluster")[[
    "Usage_kWh",
    "Lagging_Current_Reactive.Power_kVarh",
    "Leading_Current_Reactive_Power_kVarh",
    "Lagging_Current_Power_Factor",
    "CO2(tCO2)",
    "Leading_Current_Power_Factor",
]].mean().round(3)

main_profile["n"] = df_profile.groupby("cluster").size()

display(main_profile)

,Usage_kWh,Lagging_Current_Reactive.Power_kVarh,Leading_Current_Reactive_Power_kVarh,Lagging_Current_Power_Factor,CO2(tCO2),Leading_Current_Power_Factor,n
cluster,,,,,,,
0,9.961,0.574,14.695,99.422,0.003,40.951,7222
1,69.312,32.109,0.143,90.038,0.032,99.984,9548
2,4.358,5.408,0.063,58.903,0.000,99.399,11262


### 5) Avaliação da Performance dos dados

#### 5.1) Silhouette

o **Coeficiente de Silhueta** para avaliar o quanto cada ponto de dado
está bem agrupado em comparação com os outros grupos vizinhos, caso o resultado
seja abaixo de 0, ou abaixo de 0.2, não há proximidade entre os grupos.

In [ ]:
def silhouette_for(features_train, n_clusters=3, seed=42):
    """ 
    Calculo da Métrica Silhouette através da métrica 'KMeans'
    """
    km = KMeans(n_clusters=n_clusters, random_state=seed, n_init=10).fit(features_train)
    return silhouette_score(features_train, km.labels_)

# Todas as Features Numéricas
X_all = X_train_featured[
                    ['Usage_kWh', 'Lagging_Current_Reactive.Power_kVarh', 'Leading_Current_Reactive_Power_kVarh',
                     'CO2(tCO2)', 'Lagging_Current_Power_Factor', 'Leading_Current_Power_Factor',
                     'hour', 'month', 'dayofweek_num', 'NSM_sin', 'NSM_cos'
                    ]].values

X_all_scalled = StandardScaler().fit_transform(X_all)
silhouette_all = silhouette_for(features_train=X_all_scalled)


# Todas as Features que sobreviveram a curadoria
X_selected = X_train_featured[FEATURE_NUMERIC]
X_selected_scalled = StandardScaler().fit_transform(X_selected)
silhouette_selected = silhouette_for(features_train=X_selected_scalled)


# Com PCA
silhouette_pca = silhouette_for(features_train=X_train_processed)

print(f"Silhouette com TODAS as features: {silhouette_all:.4f} | Agrupamento ")
print(f"Silhouette com FEATURES selecionadas: {silhouette_selected:.4f} | Agrupamento")
print(f"Silhouette com PCA (90% var): {silhouette_pca:.4f} | Agrupamento ")

Silhouette com TODAS as features: 0.3980
Silhouette com FEATURES selecionadas: 0.3455
Silhouette com PCA (90% var): 0.3493


In [ ]:
def silhouette_with(cols, featured, n_clusters=3, seed=42):
    X = featured[cols].values
    X_scaled = StandardScaler().fit_transform(X)
    return silhouette_for(X_scaled, n_clusters=n_clusters, seed=seed)

base_cols = [
    "Usage_kWh",
    "Lagging_Current_Reactive.Power_kVarh",
    "Leading_Current_Reactive_Power_kVarh",
    "Lagging_Current_Power_Factor",
    "NSM_sin", "NSM_cos",
    "hour", "month", "dayofweek_num",
]

configs = {
    "Base (sem CO2, sem Leading_PF)":           base_cols,
    "Base + CO2":                                base_cols + ["CO2(tCO2)"],
    "Base + Leading_PF":                         base_cols + ["Leading_Current_Power_Factor"],
    "Base + CO2 + Leading_PF (todas)":           base_cols + ["CO2(tCO2)", "Leading_Current_Power_Factor"],
}

for name, cols in configs.items():
    s = silhouette_with(cols, X_train_featured)
    print(f"{name:40s} → silhouette = {s:.4f}")

Base (sem CO2, sem Leading_PF)           → silhouette = 0.3455
Base + CO2                               → silhouette = 0.3617
Base + Leading_PF                        → silhouette = 0.3717
Base + CO2 + Leading_PF (todas)          → silhouette = 0.3980


#### 5.2) ARI

Avaliar como o K-Means se comporta diante de várias sementes diferentes (ou em subamostras diferentes),
e se os clusters são os mesmos, se sim são estáveis, se mudam constantemente tem ruído.

In [ ]:
def stability(features, n_clusters=3, n_runs=20):

    labels_runs = []
    for seed in range(n_runs):
        km = KMeans(n_clusters=n_clusters, random_state=seed, n_init=1).fit(features)
        labels_runs.append(km.labels_)
    
    # Avaliação da similaridade entre 2 ou mais agrupamentos
    scores = [adjusted_rand_score(labels_runs[0], lr) for lr in labels_runs[1:]]
    return np.mean(scores), np.std(scores)


mean_all, std_all = stability(features=X_all_scalled)
mean_sel, std_sel = stability(features=X_selected_scalled)
mean_pca, std_pca = stability(features=X_train_processed)

print('Estabilidade (ARI entre as features)')
print(f"  Todas as features:    {mean_all:.3f} | {std_all:.3f}")
print(f"  Features selecionadas: {mean_sel:.3f} | {std_sel:.3f}")
print(f"  Com PCA:              {mean_pca:.3f} | {std_pca:.3f}")

Estabilidade (ARI entre as features)
  Todas as features:    1.000 | 0.000
  Features selecionadas: 0.998 | 0.001
  Com PCA:              0.999 | 0.001


#### 5.3) Testes de Remoção de Colunas

- Avaliar se cada cluster enxerga as colunas que foram removidas e se os clusters fazem sentido de domínio.
  
   * 'CO2(tCO2)' -> redundancia com a coluna 'Usage_kWh'
   * 'Leading_Current_Power_Factor' -> suspeita de constância

In [ ]:
# Treinamento com as features selecionadas
km_sel = KMeans(n_clusters=3, random_state=42, n_init=10).fit(X_selected_scalled)
labels_sel = km_sel.labels_

# Coloca as labels novamente com as features antes removidas
df_check = X_train_featured.copy()
df_check['cluster'] = labels_sel

# Calcula a variabilidade das features removidas por cluster
for feature in ['CO2(tCO2)', 'Leading_Current_Power_Factor']:

    mean_per_cluster = df_check.groupby("cluster")[feature].mean()
    variance = mean_per_cluster.max() - mean_per_cluster.min()
    std_global = df_check[feature].std()
    ratio = variance/std_global if std_global > 0 else 0

    print(f"{feature}:")
    print(f"- médias por cluster: {mean_per_cluster.round(3).to_dict()}")
    print(f"- variação entre clusters: {variance:.4f}")
    print(f"- std entre clusters: {std_global:.4f}")
    print(f"- razão variação/std global: {ratio:.3f}\n")


CO2(tCO2):
- médias por cluster: {0: 0.003, 1: 0.0, 2: 0.032}
- variação entre clusters: 0.0312
- std entre clusters: 0.0164
- razão variação/std global: 1.909

Leading_Current_Power_Factor:
- médias por cluster: {0: 41.088, 1: 99.398, 2: 99.986}
- variação entre clusters: 58.8980
- std entre clusters: 30.3782
- razão variação/std global: 1.939



### 6) Salvamento de Dados

In [ ]:
# Processamento de dados (treino e teste)
np.save(PROCESSED / "x_train.npy", X_train_processed)
np.save(PROCESSED / "x_test.npy", X_test_processed)

y_train.to_csv(PROCESSED / "y_train.csv", index=False)
y_test.to_csv(PROCESSED / "y_test.csv", index=False)


# DataFrames Enriquecidos
train_enriched = X_train.copy()
test_enriched = X_test.copy()

for i in range(X_train_processed.shape[1]):
    train_enriched[f"PC{i+1}"] = X_train_processed[:, i]
    test_enriched [f"PC{i+1}"] = X_test_processed[:, i]

train_enriched.to_csv(PROCESSED / "X_train_enriched.csv", index=False)
test_enriched.to_csv(PROCESSED / "X_test_enriched.csv", index=False)


# Pipeline Treinado
joblib.dump(pipeline, MODELS / "preprocessor.joblib")


# Definição das Features
feature_data = {
    "raw_file": str(RAW),
    
    "feature_numeric": FEATURE_NUMERIC,
    "feature_categoric": FEATURE_CLUSTERS_CATEGORICAL,
    "feature_discarded": FEATURE_DISCARTED,
    "feature_deleted": FEATURE_DELETED,
    "target": target_column,

    "pca_components": [f"PC{i+1}" for i in range(components)],
    "one_hot_encoding_features": list(
            pipeline.named_steps["prep"]\
            .named_transformers_["cat"]\
            .get_feature_names_out(FEATURE_CLUSTERS_CATEGORICAL)
    ),

    "split_strategy": "temporal_80_20",
    "n_rows_total": int(len(df)),
    "n_train": int(len(X_train)),
    "n_test": int(len(X_test)),
    "train_date_range": [str(X_train["date"].min()), str(X_train["date"].max())],
    "test_date_range":  [str(X_test["date"].min()),  str(X_test["date"].max())],
}

(METADATA / "metadata.json").write_text(
    json.dumps(feature_data, indent=2, ensure_ascii=False),
    encoding="utf-8"
)

1083